# 04 — Aerodynamic Performance

Parabolic drag polar at the design mass: characteristic speeds,
power-required curve and L/D. Every optimum speed is checked against
stall — on a low-AR flying wing with a reflexed section the theoretical
best-endurance speed is often below stall, so the aircraft is
stall-limited rather than drag-limited.

Uses the design target mass; swap in `balance_summary['auw_kg']` from
notebook 03 once the mass budget is real.


In [ ]:
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "flying_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "flying_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
with open(REPO_ROOT / "mission" / "mission_params.yaml") as f:
    mission = yaml.safe_load(f)
with open(PLATFORM_DIR / "design_point.yaml") as f:
    design = yaml.safe_load(f)
with open(PLATFORM_DIR / "airframe_params.yaml") as f:
    airframe = yaml.safe_load(f)

from uavdesign.atmosphere import G, isa_density, wing_loading_g_dm2, wing_loading_n_m2
from uavdesign.aero import Wing, DragPolar, FixedWingAero

alt_m = mission["mission"]["altitude_asl_m"]
rho = isa_density(alt_m, mission["environment"]["temperature_offset_c"])
wing = Wing.from_dict(airframe["wing"])
aero = design["aero"]
perf = design["performance"]
mass_kg = design["design_point"]["target_auw_kg"]
static_margin = design["design_point"]["static_margin"]
polar = DragPolar.from_oswald(aero["cd0"], wing.aspect_ratio, aero["oswald_e"], aero["cl_max"])

print(f"Site: {alt_m} m ASL, rho = {rho:.4f} kg/m^3")
print(f"Wing: S = {wing.area_dm2:.2f} dm^2, b = {wing.span_mm:.0f} mm, AR = {wing.aspect_ratio:.2f}")
print(f"Polar: CD0 = {polar.cd0}, k = {polar.k:.4f}, CLmax = {polar.cl_max}")


In [ ]:
from uavdesign import propulsion as prop

ac = FixedWingAero(mass_kg, wing.area_m2, polar, rho)
pp = design["propulsion"]
vs = ac.stall_speed()
limited = ac.stall_limited(margin=1.2)
v_endurance = max(ac.v_min_power, 1.2 * vs)
v_range = max(ac.v_max_ld, 1.2 * vs)

def flag(name):
    return "  <-- below 1.2 Vs, stall-limited" if limited[name] else ""

print(f"Wing loading:            {ac.wing_loading_n_m2:.1f} N/m^2 "
      f"({wing_loading_g_dm2(ac.wing_loading_n_m2):.1f} g/dm^2)")
print(f"Stall speed (CLmax {polar.cl_max}): {vs:5.1f} m/s ({vs*3.6:.0f} km/h)")
print(f"Min-power speed:         {ac.v_min_power:5.1f} m/s  (CL {polar.cl_min_power:.2f}){flag('v_min_power')}")
print(f"Best-L/D speed:          {ac.v_max_ld:5.1f} m/s  (CL {polar.cl_max_ld:.2f}){flag('v_max_ld')}")
print(f"L/D max (unconstrained): {polar.ld_max:5.1f}")
print(f"-> best endurance speed: {v_endurance:5.1f} m/s, best range speed: {v_range:5.1f} m/s")
print(f"Cruise ({perf['cruise_speed_m_s']} m/s):          CL {float(ac.cl(perf['cruise_speed_m_s'])):.2f}, "
      f"L/D {float(ac.lift_to_drag(perf['cruise_speed_m_s'])):.1f}")


## Power required and L/D vs airspeed


In [ ]:
v = np.linspace(0.8 * vs, 30, 300)
p_aero = ac.aero_power_w(v)
p_shaft = p_aero / pp["prop_efficiency"]
p_elec = prop.electrical_power_w(p_shaft, pp["motor_efficiency"], pp["esc_efficiency"]) \
    + pp["avionics_power_w"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.plot(v, p_aero, label="aero  D*V")
ax1.plot(v, p_shaft, label="shaft  / eta_prop")
ax1.plot(v, p_elec, label="electrical  / eta_motor*eta_esc + avionics")
for a in (ax1, ax2):
    a.axvspan(v[0], vs, color="C3", alpha=0.12)
    a.axvline(1.2 * vs, color="C3", ls=":", lw=1)
    a.axvline(perf["cruise_speed_m_s"], color="k", ls="--", lw=1)
    a.set_xlabel("Airspeed [m/s]")
    a.grid(alpha=0.3)
ax1.set_ylabel("Power [W]")
ax1.set_ylim(0, float(p_elec.max()))
ax1.legend(fontsize=8)
ax1.set_title("Power required (shaded: below stall; dotted: 1.2 Vs)")
ax2.plot(v, ac.lift_to_drag(v), color="C2")
ax2.set_ylabel("L/D")
ax2.set_title("Lift-to-drag ratio")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_power_required.png", dpi=140)
plt.show()


## Sensitivity to CLmax

The online calculator assumes CLmax = 1.0 at sea level. A trimmed,
reflexed flying-wing section is closer to 0.6-0.8.


In [ ]:
import pandas as pd
from uavdesign.atmosphere import RHO0

rows = []
for cl in aero["cl_max_sweep"]:
    p = DragPolar(polar.cd0, polar.k, cl)
    here = FixedWingAero(mass_kg, wing.area_m2, p, rho)
    sea = FixedWingAero(mass_kg, wing.area_m2, p, RHO0)
    rows.append({
        "CLmax": cl,
        "Vs sea level [km/h]": sea.stall_speed() * 3.6,
        f"Vs {alt_m} m [km/h]": here.stall_speed() * 3.6,
        "launch speed needed [km/h]": here.stall_speed() * perf["launch_margin"] * 3.6,
        "endurance-limited by stall": here.stall_limited()["v_min_power"],
    })
pd.DataFrame(rows).round(1)


## Outputs handed to notebook 05


In [ ]:
aero_summary = {
    "stall_speed_m_s": vs,
    "v_best_endurance_m_s": v_endurance,
    "v_best_range_m_s": v_range,
    "cruise_drag_n": float(ac.drag_n(perf["cruise_speed_m_s"])),
    "cruise_aero_power_w": float(ac.aero_power_w(perf["cruise_speed_m_s"])),
}
aero_summary
